<a href="https://colab.research.google.com/github/mrayhanz/244107020027-Machine-Learning/blob/main/Minggu%2015/MRayhanZamzami_JS15_P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDENTITAS DIRI

Nama : Muhammad Rayhan Zamzami

NIM : 244107020027

Kelas : TI-3G

Absen : 19

# PRAKTIKUM 2 - Deployment ke Hugging Face (Docker + Flask)

In [1]:
import os
import shutil

project_dir = 'daynight-project'
os.makedirs(project_dir, exist_ok=True)

for fname in ['day_night_model.h5', 'scaler.pkl']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(project_dir, fname))
        print('copied', fname)
    else:
        print('WARNING: file belum ada ->', fname)

copied day_night_model.h5
copied scaler.pkl


In [2]:
app_code = 'import os\nimport numpy as np\nimport cv2\nimport pickle\nimport tensorflow as tf\nfrom flask import Flask, request, render_template_string\nfrom skimage.feature import hog\n\napp = Flask(__name__)\n\nMODEL_PATH = "day_night_model.h5"\nSCALER_PATH = "scaler.pkl"\n\ntry:\n    model = tf.keras.models.load_model(MODEL_PATH)\n    with open(SCALER_PATH, "rb") as f:\n        scaler = pickle.load(f)\n    print("System Loaded Successfully")\nexcept Exception as e:\n    print(f"Error loading system: {e}")\n\ndef preprocess_image(image_bytes):\n    nparr = np.frombuffer(image_bytes, np.uint8)\n    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)\n    img = cv2.resize(img, (256, 256))\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8,8),\n                   cells_per_block=(2,2), block_norm="L2-Hys",\n                   visualize=False, feature_vector=True)\n    return scaler.transform(hog_feat.reshape(1, -1))\n\n@app.route("/", methods=["GET"])\ndef home():\n    html = """<div style="text-align:center; padding:50px;">\n        <h1>Day vs Night Classifier</h1>\n        <form action="/predict" method="post" enctype="multipart/form-data">\n            <input type="file" name="file" required><br><br>\n            <button type="submit">Prediksi</button>\n        </form>\n    </div>"""\n    return render_template_string(html)\n\n@app.route("/predict", methods=["POST"])\ndef predict():\n    try:\n        file = request.files["file"]\n        data = preprocess_image(file.read())\n        prediction = model.predict(data)[0][0]\n        label = "Day (Siang)" if prediction > 0.5 else "Night (Malam)"\n        return f"<h2 style=\'text-align:center\'>Hasil: {label}</h2><center><a href=\'/\'>Kembali</a></center>"\n    except Exception as e:\n        return f"Error: {e}"\n\nif __name__ == "__main__":\n    app.run(host="0.0.0.0", port=7860)\n'

with open(os.path.join(project_dir, 'app.py'), 'w', encoding='utf-8') as f:
    f.write(app_code)
print('app.py created')

app.py created


In [3]:
requirements = '''flask
tensorflow-cpu
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn
'''

with open(os.path.join(project_dir, 'requirements.txt'), 'w', encoding='utf-8') as f:
    f.write(requirements)

dockerfile = '''FROM python:3.9-slim

RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

WORKDIR /app

COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

COPY --chown=user . /app

EXPOSE 7860
CMD ["python", "app.py"]
'''

with open(os.path.join(project_dir, 'Dockerfile'), 'w', encoding='utf-8') as f:
    f.write(dockerfile)

readme = '''---
title: Daynight Classifier
emoji: 🌙
colorFrom: blue
colorTo: indigo
sdk: docker
pinned: false
---

# Day vs Night Classifier
Deployed with Flask + Docker on Hugging Face Spaces.
'''

with open(os.path.join(project_dir, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme)

print('requirements.txt, Dockerfile, README.md created')

requirements.txt, Dockerfile, README.md created


In [4]:
print('Isi folder deployment:')
for f in sorted(os.listdir(project_dir)):
    size = os.path.getsize(os.path.join(project_dir, f))
    print(f' - {f} ({size} bytes)')

shutil.make_archive('daynight-project', 'zip', project_dir)
print('Arsip deployment dibuat: daynight-project.zip')

Isi folder deployment:
 - Dockerfile (415 bytes)
 - README.md (196 bytes)
 - app.py (1900 bytes)
 - day_night_model.h5 (53273328 bytes)
 - requirements.txt (92 bytes)
 - scaler.pkl (830743 bytes)


Arsip deployment dibuat: daynight-project.zip


In [5]:
print('Langkah deploy ke Hugging Face:')
print('1. Buat Space baru (SDK: Docker, blank template).')
print('2. git clone https://huggingface.co/spaces/USERNAME/NAMA_SPACE')
print('3. Salin seluruh isi folder daynight-project ke folder hasil clone.')
print('4. git add . && git commit -m "Deploy day/night classifier" && git push')

Langkah deploy ke Hugging Face:
1. Buat Space baru (SDK: Docker, blank template).
2. git clone https://huggingface.co/spaces/USERNAME/NAMA_SPACE
3. Salin seluruh isi folder daynight-project ke folder hasil clone.
4. git add . && git commit -m "Deploy day/night classifier" && git push
